<a href="https://colab.research.google.com/github/zhangling297/deep-learning-with-python-notebooks/blob/master/Cs599_Assignment1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. A regular perceptron
Created by Ling Zhang
Date: 02-15-2026

In [ ]:
import re
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd
df = pd.read_csv("/content/IMDB Dataset.csv")
df.head()


In [ ]:
# Data cleaning
review_yes = list(df[df['sentiment'] == 'positive']['review'].apply(str))
review_no = list(df[df['sentiment'] == 'negative']['review'].apply(str))
texts = review_yes + review_no

#Yes = Positive (+1), No = Negative (-1)
y = np.array([1] * len(review_yes) + [-1] * len(review_no), dtype=int)

#Tokenization / Clearning
WORD_RE = re.compile(r"[a-zA-Z']+")

def normalize(text: str) ->str:
  #Fix hyphenation across line breaks: "per-\nformances" -> "performance"
  text = text.replace("-\n", "")
  return text.replace("\n", " ")

def tokenize(text: str):
  text = normalize(text).lower()
  return WORD_RE.findall(text)

# Vectorization (Bag-of-Words)

def build_vocab(texts):
  counts = Counter()
  for t in texts:
    counts.update(tokenize(t))
  #Stable ordering
  vocab = {w: i for i, w in enumerate(sorted(counts.keys()))}
  return vocab

def vectorize(text: str, vocab: dict) -> np.ndarray:
  x = np.zeros(len(vocab), dtype=int)
  for w in tokenize(text):
    idx = vocab.get(w)
    if idx is not None:
      x[idx] += 1
  return x

# Perceptron matching pseudocode
YES, NO = 1, -1

def decision(xi: np.ndarray, w: np.ndarray, b: int) -> int:
  """Return YES if w * xi + b >= 0, else NO"""
def perceptron_train(X: np.ndarray, y: np.ndarray, max_epochs: int = 1000):
  """
  Match 2nd Algorith:

  w = 0
  b = 0
  while not converged:

    for each xi:
      d = dcision(xi, w, b)
      if d == yi: continue
      else if yi == Yes and d == No: b = b+1, w = w + xi
      else if yi == No, and d == Yes: b = b-1, w = w - xi
  """
  w = np.zeros( X.shape[1], dtype = int) # w = 0
  b = 0                                  # b = 0

  for epoch in range(max_epochs):
    mistakes = 0 # Corrected from 'o'
    for xi, yi in zip(X, y):
      d = decision(xi, w, b)

      if d == yi:
        continue

      mistakes += 1 # This should be outside the if/else if block, and incremented only when a mistake is made.
      if yi == YES and d == NO:
        b = b + 1
        w = w + xi
      elif yi == NO and d == YES: # Corrected 'else' to 'elif'
        b = b - 1
        w = w - xi

    # Converged means ( no mistakes in full pass)
    if mistakes == 0:
      #print(f"Converged at epoch {epoch+1}")
      break # Break needs to be inside the loop it controls (for epoch

  return w, b

In [ ]:
vocab = build_vocab(texts)
X = np.vstack([vectorize(t, vocab) for t in texts])
w, b = perceptron_train(X, y, max_epochs = 100)
preds = np.array([decision(xi, w, b) for xi in X], dtype = int)
print("Predictions (Yes = +1 = positive), No = -1 = negative):", preds.tolist())
print("True labels:", y.tolist())
print("w nonzero:", int((w !=0).sum(), " / ", w.size))
print("b:", b)
